# Étape 4 bis — Fine-tuning partiel de ResNet-50

Jusqu'ici, ResNet-50 restait gelé et seule une tête de classification apprenait, sur les embeddings calculés une fois pour toutes. Ce notebook teste l'alternative : **réentraîner le dernier bloc convolutif (`layer4`) avec la tête**, à partir des images, avec ou sans augmentation de données.

Précautions, avec environ 15 millions de paramètres entraînables (64 % du réseau) pour 55 images par pli :
- `layer1` à `layer3` restent gelés, et les BatchNorm conservent les statistiques d'ImageNet (sinon, elles se recalculeraient sur des lots de 16 IRM, même dans les couches gelées) ;
- deux taux d'apprentissage : 0,00001 pour ajuster `layer4` sans détruire ce qu'il a appris sur ImageNet, 0,001 pour la tête, qui part de zéro ;
- arrêt anticipé sur la perte de validation (patience de 10 epochs, plafond de 100) ;
- augmentation sur l'entraînement uniquement (retournement gauche-droite, rotation de ± 10°), jamais sur la validation ni le test ;
- les mêmes plis que dans le notebook 04 (le découpage stratifié ne dépend que des labels et de la graine), pour une comparaison pli par pli avec les têtes gelées.

In [1]:
import copy
import numpy as np
import pandas as pd
import torch
from torch import nn
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import recall_score, precision_score, fbeta_score

inventaire = pd.read_csv("../data/inventaire.csv")
masque_train = (inventaire["split"] == "train").to_numpy()
masque_test = (inventaire["split"] == "test").to_numpy()

y_fort = (inventaire.loc[masque_train, "label"] == "cancer").astype(int).to_numpy()
y_test = (inventaire.loc[masque_test, "label"] == "cancer").astype(int).to_numpy()

# Les 99 images labellisées sont chargées une seule fois en mémoire (elles sont peu nombreuses)
images_fort = [Image.open(c).convert("RGB") for c in inventaire.loc[masque_train, "chemin"]]
images_test = [Image.open(c).convert("RGB") for c in inventaire.loc[masque_test, "chemin"]]

# Les MÊMES plis que dans le notebook 04 : le découpage ne dépend que des labels et de la graine
plis = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42).split(np.zeros(len(y_fort)), y_fort))

appareil = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(len(images_fort), "images fortes |", len(images_test), "images de test |", len(plis), "plis | appareil :", appareil)
print("Pli 0 :", len(plis[0][0]), "train /", len(plis[0][1]), "validation, dont", y_fort[plis[0][1]].sum(), "cancers")


69 images fortes | 30 images de test | 15 plis | appareil : mps
Pli 0 : 55 train / 14 validation, dont 7 cancers


In [2]:
normalisation = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])   # ImageNet

# Validation et test : le prétraitement habituel, toujours identique
transfo_eval = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalisation,
])

# Entraînement : le même, avec en plus des variations aléatoires, différentes à chaque epoch
transfo_augmentee = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),     # retournement gauche-droite, une fois sur deux
    transforms.RandomRotation(10),         # rotation aléatoire entre -10° et +10°
    transforms.ToTensor(),
    normalisation,
])

# Sans augmentation, les tenseurs ne changent jamais : on les calcule une seule fois
tenseurs_fort = torch.stack([transfo_eval(img) for img in images_fort])     # (69, 3, 224, 224)
tenseurs_test = torch.stack([transfo_eval(img) for img in images_test])     # (30, 3, 224, 224)
print(tenseurs_fort.shape, tenseurs_test.shape)


torch.Size([69, 3, 224, 224]) torch.Size([30, 3, 224, 224])


In [3]:
def creer_modele(graine=0):
    """ResNet-50 pré-entraîné : tout gelé sauf layer4, avec une nouvelle tête à 2 sorties."""
    torch.manual_seed(graine)                                   # la seule part de hasard : l'initialisation de la tête
    modele = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)   # mêmes poids que pour les embeddings
    for p in modele.parameters():
        p.requires_grad = False                                 # 1) on gèle tout
    for p in modele.layer4.parameters():
        p.requires_grad = True                                  # 2) on dégèle layer4
    modele.fc = nn.Linear(2048, 2)                              # 3) nouvelle tête, entraînable par défaut
    return modele.to(appareil)

def mode_entrainement(modele):
    """Met le modèle en mode entraînement, mais garde les BatchNorm sur les statistiques d'ImageNet."""
    modele.train()
    for couche in modele.modules():
        if isinstance(couche, nn.BatchNorm2d):
            couche.eval()

modele = creer_modele()
entrainables = sum(p.numel() for p in modele.parameters() if p.requires_grad)
total = sum(p.numel() for p in modele.parameters())
print(f"Paramètres entraînables : {entrainables:,} sur {total:,} ({entrainables / total:.0%})")


Paramètres entraînables : 14,968,834 sur 23,512,130 (64%)


In [4]:
def lot_augmente(indices):
    """Tenseurs augmentés des images fortes demandées : un nouveau tirage aléatoire à chaque appel."""
    return torch.stack([transfo_augmentee(images_fort[i]) for i in indices])

def predire_proba(modele, tenseurs, taille_lot=32):
    """Probabilité « cancer » pour chaque image, par lots (mode évaluation, sans gradients)."""
    modele.eval()
    sorties = []
    with torch.no_grad():
        for debut in range(0, len(tenseurs), taille_lot):
            scores = modele(tenseurs[debut:debut + taille_lot].to(appareil))
            sorties.append(torch.softmax(scores, dim=1)[:, 1].cpu())
    return torch.cat(sorties).numpy()

def mesurer(proba, y, seuil=0.5):
    """Rappel, précision et F2 au seuil donné."""
    pred = (proba >= seuil).astype(int)
    return {"rappel": recall_score(y, pred, zero_division=0),
            "précision": precision_score(y, pred, zero_division=0),
            "F2": fbeta_score(y, pred, beta=2, zero_division=0)}

In [5]:
def entrainer_ft(modele, idx_tr, idx_val=None, epoques=100, augmentation=True,
                 lr_layer4=1e-5, lr_tete=1e-3, weight_decay=1e-4, taille_lot=16, patience=10, graine=0):
    """Fine-tuning de layer4 + tête sur les images fortes idx_tr.
    Avec idx_val : arrêt anticipé sur la perte de validation, retour aux meilleurs poids,
    et renvoie le nombre d'epochs retenu."""
    optimiseur = torch.optim.Adam([
        {"params": modele.layer4.parameters(), "lr": lr_layer4},   # petits pas : ajuster sans détruire
        {"params": modele.fc.parameters(), "lr": lr_tete},         # pas normaux : la tête part de zéro
    ], weight_decay=weight_decay)
    perte_fn = nn.CrossEntropyLoss()
    generateur = torch.Generator().manual_seed(graine)            # ordre des lots reproductible
    torch.manual_seed(graine)                                      # tirages de l'augmentation reproductibles
    y_tr = torch.tensor(y_fort[idx_tr], dtype=torch.long)

    avec_validation = idx_val is not None
    if avec_validation:
        X_val = tenseurs_fort[idx_val].to(appareil)                # jamais augmentées
        y_val = torch.tensor(y_fort[idx_val], dtype=torch.long).to(appareil)
        meilleure_perte, meilleure_epoque = float("inf"), 0
        meilleurs_poids = copy.deepcopy(modele.state_dict())

    for epoque in range(epoques):
        mode_entrainement(modele)                                  # à chaque epoch, BatchNorm figées
        ordre = torch.randperm(len(idx_tr), generator=generateur)
        for debut in range(0, len(idx_tr), taille_lot):
            pos = ordre[debut:debut + taille_lot]                  # positions dans idx_tr
            indices = idx_tr[pos.numpy()]                          # numéros des images fortes
            X = lot_augmente(indices) if augmentation else tenseurs_fort[indices]
            perte = perte_fn(modele(X.to(appareil)), y_tr[pos].to(appareil))
            optimiseur.zero_grad()
            perte.backward()
            optimiseur.step()

        if avec_validation:
            modele.eval()
            with torch.no_grad():
                perte_val = perte_fn(modele(X_val), y_val).item()
            if perte_val < meilleure_perte:
                meilleure_perte, meilleure_epoque = perte_val, epoque
                meilleurs_poids = copy.deepcopy(modele.state_dict())
            elif epoque - meilleure_epoque >= patience:
                break

    if avec_validation:
        modele.load_state_dict(meilleurs_poids)
        return meilleure_epoque + 1

In [6]:
%%time
idx_tr, idx_val = plis[0]
modele = creer_modele(graine=0)
nb_epoques = entrainer_ft(modele, idx_tr, idx_val, augmentation=True, graine=0)
print("Epochs retenues :", nb_epoques, "|", mesurer(predire_proba(modele, tenseurs_fort[idx_val]), y_fort[idx_val]))

Epochs retenues : 56 | {'rappel': 1.0, 'précision': 1.0, 'F2': 1.0}
CPU times: user 11.3 s, sys: 2.54 s, total: 13.8 s
Wall time: 54.7 s


In [7]:
%%time
plis_cv = plis[:5]          # première répétition : chaque image passe une fois en validation
lignes = []
for augmentation in [True, False]:
    for num_pli, (idx_tr, idx_val) in enumerate(plis_cv):
        modele = creer_modele(graine=num_pli)
        nb_epoques = entrainer_ft(modele, idx_tr, idx_val, augmentation=augmentation, graine=num_pli)
        lignes.append({"augmentation": augmentation, "pli": num_pli, "époques": nb_epoques,
                       **mesurer(predire_proba(modele, tenseurs_fort[idx_val]), y_fort[idx_val])})
        print(f"augmentation={augmentation} · pli {num_pli} · {nb_epoques} epochs · F2 {lignes[-1]['F2']:.3f}")

ft = pd.DataFrame(lignes)
ft.groupby("augmentation")[["F2", "rappel", "précision", "époques"]].agg(["mean", "std"]).round(3)

augmentation=True · pli 0 · 56 epochs · F2 1.000
augmentation=True · pli 1 · 14 epochs · F2 0.972
augmentation=True · pli 2 · 4 epochs · F2 0.857
augmentation=True · pli 3 · 22 epochs · F2 1.000
augmentation=True · pli 4 · 5 epochs · F2 0.882
augmentation=False · pli 0 · 100 epochs · F2 1.000
augmentation=False · pli 1 · 13 epochs · F2 0.972
augmentation=False · pli 2 · 3 epochs · F2 0.972
augmentation=False · pli 3 · 91 epochs · F2 1.000
augmentation=False · pli 4 · 5 epochs · F2 0.882
CPU times: user 57 s, sys: 16.1 s, total: 1min 13s
Wall time: 5min 45s


F2        rappel        précision        époques        
               mean    std   mean    std      mean    std    mean     std
augmentation                                                             
False         0.965  0.048  0.971  0.064     0.950  0.068    42.4  48.722
True          0.942  0.068  0.943  0.078     0.946  0.074    20.2  21.312

Sur les 5 plis de la première répétition (chaque image passe une fois en validation ; 15 plis auraient demandé près d'une demi-heure de calcul), le fine-tuning obtient un F2 de 0,965 ± 0,048 sans augmentation et de 0,942 ± 0,068 avec. Comparé pli par pli aux têtes gelées du notebook 04, il fait mieux que la tête linéaire (+ 0,10, 3 plis meilleurs, aucun moins bon), mais **fait jeu égal avec la tête gelée retenue** : 4 plis identiques sur 5. Avec 5 plis d'environ 14 images, un pli bascule pour une ou deux images : cet écart n'est pas une tendance.

In [8]:
grille = pd.read_csv("../data/grille_cv.csv")

def gelee(lr, l2, tete):
    """F2 par pli d'une tête gelée du notebook 04, sur les 5 plis de la première répétition."""
    sel = grille[(grille["lr"] == lr) & (grille["L2"] == l2) & (grille["tête"] == tete) & (grille["pli"] < 5)]
    return sel.set_index("pli")["F2"]

references = {
    "tête gelée linéaire (baseline)": gelee(1e-3, 1e-4, "linéaire"),
    "tête gelée 256 + dropout 0,5 (retenue)": gelee(1e-3, 1e-3, "256 + dropout 0.5"),
}
for augmentation in [True, False]:
    f2_ft = ft[ft["augmentation"] == augmentation].set_index("pli")["F2"]
    for nom, reference in references.items():
        ecart = f2_ft - reference
        print(f"fine-tuning (augmentation={augmentation}) contre {nom} : écart {ecart.mean():+.3f}"
              f" | mieux {(ecart > 0).sum()} · égal {(ecart == 0).sum()} · moins bien {(ecart < 0).sum()}")
    print()

fine-tuning (augmentation=True) contre tête gelée linéaire (baseline) : écart +0.095 | mieux 3 · égal 2 · moins bien 0
fine-tuning (augmentation=True) contre tête gelée 256 + dropout 0,5 (retenue) : écart +0.024 | mieux 1 · égal 4 · moins bien 0

fine-tuning (augmentation=False) contre tête gelée linéaire (baseline) : écart +0.118 | mieux 3 · égal 2 · moins bien 0
fine-tuning (augmentation=False) contre tête gelée 256 + dropout 0,5 (retenue) : écart +0.047 | mieux 1 · égal 4 · moins bien 0



**Décision prise avant d'ouvrir le test.** En validation croisée, le fine-tuning de `layer4` fait jeu égal avec la tête gelée retenue (4 plis identiques sur 5, le cinquième meilleur). À égalité, le modèle le plus simple reste la référence : la tête entraînée sur ResNet gelé. Le fine-tuning, dans sa configuration la plus simple (sans augmentation, qui n'apporte rien), est évalué une seule fois sur le test, à titre d'information : ce résultat ne servira à aucun choix.

In [9]:
avec = ft[ft["augmentation"]].set_index("pli")
sans = ft[~ft["augmentation"]].set_index("pli")
ecart = avec["F2"] - sans["F2"]
print(f"Avec - sans augmentation : {ecart.mean():+.3f}"
      f" | mieux {(ecart > 0).sum()} · égal {(ecart == 0).sum()} · moins bien {(ecart < 0).sum()}")
pd.DataFrame({"epochs avec": avec["époques"], "epochs sans": sans["époques"]})

Avec - sans augmentation : -0.023 | mieux 0 · égal 4 · moins bien 1


,epochs avec,epochs sans
pli,,
0,56,100
1,14,13
2,4,3
3,22,91
4,5,5


L'augmentation n'est jamais meilleure (4 plis égaux, 1 moins bon) et arrête l'entraînement plus tôt. Hypothèses : l'arrêt anticipé protège déjà contre le surapprentissage, et un retournement ou une petite rotation ne reproduisent pas les variations qui distinguent un cancer d'une image normale. Elle n'est pas retenue.

In [10]:
%%time
from sklearn.metrics import roc_auc_score

EPOQUES_FT = int(sans["époques"].median())       # nombre d'epochs médian en validation croisée, sans augmentation
print("Epochs retenues :", EPOQUES_FT)

resultats_ft, probas_ft = [], []
for graine in range(3):
    modele = creer_modele(graine=graine)
    entrainer_ft(modele, np.arange(len(y_fort)), epoques=EPOQUES_FT, augmentation=False, graine=graine)   # les 69 images
    proba = predire_proba(modele, tenseurs_test)
    probas_ft.append(proba)
    resultats_ft.append({"graine": graine, **mesurer(proba, y_test), "AUC": roc_auc_score(y_test, proba)})
display(pd.DataFrame(resultats_ft).drop(columns="graine").agg(["mean", "std"]).round(3))

# Les deux cancers ratés par toutes les méthodes du notebook 04
chemins_test = inventaire.loc[masque_test, "chemin"].to_numpy()
for debut_nom in ["7e3f6acd", "9973120d"]:
    i = next(k for k, c in enumerate(chemins_test) if debut_nom in c)
    print(debut_nom, "→ probabilité de cancer selon les 3 graines :", [round(float(p[i]), 3) for p in probas_ft])


Epochs retenues : 13


,rappel,précision,F2,AUC
mean,0.867,0.929,0.878,0.953
std,0.000,0.000,0.000,0.005


7e3f6acd → probabilité de cancer selon les 3 graines : [0.027, 0.024, 0.052]
9973120d → probabilité de cancer selon les 3 graines : [0.004, 0.003, 0.003]
CPU times: user 6.6 s, sys: 2.15 s, total: 8.74 s
Wall time: 35.9 s


Sur le test (69 images d'entraînement, 13 epochs, 3 graines), le fine-tuning obtient un rappel de 0,867, une précision de 0,929, un F2 de 0,878 et une AUC de 0,953 : il trouve les mêmes 13 cancers sur 15 que les têtes gelées, avec une fausse alerte de plus. Les deux cancers ratés restent sous 5 % de probabilité : réentraîner le dernier bloc ne suffit pas à les rattraper.

Comme décidé avant d'ouvrir le test, **la tête sur ResNet gelé reste le modèle de référence** : à performances égales, elle est plus simple, s'entraîne en quelques secondes sur des embeddings calculés une seule fois, et se réentraîne facilement à l'échelle.

## Conclusion de l'étape 4 bis

| Modèle | Validation croisée | Test (rappel / précision / F2 / AUC) |
|---|---|---|
| Tête gelée retenue (256 + dropout 0,5) | référence | 0,867 / 0,957 / 0,883 / 0,982 |
| Fine-tuning `layer4`, sans augmentation | égalité (4 plis identiques sur 5) | 0,867 / 0,929 / 0,878 / 0,953 |
| Fine-tuning `layer4`, avec augmentation | jamais meilleur que sans | non évalué |

Le fine-tuning partiel et l'augmentation de données ont été testés avec le même protocole que les têtes gelées. Aucun gain stable : le modèle gelé est conservé. Avec 69 images, la limite tient davantage à la quantité de données et à la taille du test qu'au choix d'entraîner ou non le réseau.